In [1]:
import random
import torch
import io
import pyarrow as pa
import os
import copy
from sacred import Experiment
from PIL import Image
from tqdm import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
# from refer import REFER

from renaissance.modules.heads import Pooler

from torch.utils.data import DataLoader
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule

from torch.optim import AdamW

from transformers import ElectraTokenizer
from transformers import AutoModel, AutoModelForSequenceClassification


from renaissance.transforms import keys_to_transforms
from renaissance.config import ex
from renaissance.modules import RenaissanceTransformer
from renaissance.datamodules.multitask_datamodule import MTDataModule
from renaissance.datasets.base_dataset import BaseDataset

In [4]:
def _loss_names(d):
    ret = {
        "itm": 0,
        "mlm": 0,
        "mpp": 0,
        "vqa": 0,
        "vcr": 0,
        "vcr_qar": 0,
        "nlvr2": 0,
        "irtr": 0,
        "contras": 0,
        "snli": 0,
        "ref": 0,
        "mrpc" : 0,
        "rte" : 0,
        'wnli' : 0,
        'sst2' : 0,
        'qqp' : 0,
        'qnli' : 0,
        'mnli' : 0,
        'cola' : 0,
        'cifar10' : 0
    }
    ret.update(d)
    return ret


_config = {  
    "exp_name":"dev_optimizer",
    "seed" : 2244,
    "datasets" : ["coco", "vg"],
    "loss_names" : _loss_names({"itm": 1, "mlm": 1}),
    "batch_size" : 256,  # this is a desired batch size; pl trainer will accumulate gradients when per step batch is smaller.
    "per_gpu_batchsize" : 16,  # you should define this manually with per_gpu_batch_size=#
    "eval_batch_size" : 32,
    
    # Path to .ckpt file for fine-tuning or testing
    # "load_path" : "result/test_mlm_itm_a_seed0_is224_ps16_bs32_pgbs2_ts5/version_0/checkpoints/last.ckpt",
    "load_path" : "",
    # Path to .ckpt file for resuming training from previous checkpoint
    "resume_from" : None,

    # Model Type Setting
    "model_type" : "two-tower", # Supports ['one-tower', 'two-tower]
    
    #### One Tower Settings ####
    # one-tower settings will be ignored if unless model_type = "one-tower"
    # Text Setting
    "encoder" : "google/electra-small-discriminator",
    "pooler_type" : 'double', # Supports ['single', 'double']
    "tokenizer" : "bert-base-uncased",

    # Transformer Setting
    # Train encoder model from scratch
    "random_init_encoder" : False,
    ## Manual Configuration
    "encoder_manual_configuration" : False,
    "hidden_size" : 192,
    "num_heads" : 4,
    "num_layers" : 12,
    "mlp_ratio" : 4,
    "drop_rate" : 0.1,
    "embedding_size" : 96,

    #### Two Tower Settings ####
    ### Image Encoder settings
    "image_encoder" : "facebook/deit-tiny-patch16-224",
    ## Train encoder model from scratch
    "random_init_vision_encoder" : False,
    ## Manual Configure Image Encoder
    "image_encoder_manual_configuration" : False,
    ## Manual Configuration
    "image_encoder_hidden_size" : 192,
    "image_encoder_num_heads" : 4,
    "image_encoder_num_layers" : 12,
    "image_encoder_mlp_ratio" : 4,
    "image_encoder_drop_rate" : 0.1,
    "image_encoder_embedding_size" : 128,
    "image_size" : 224,
    "original_image_size" : 224, # Image size model is pretrained with, used in fine-tuning and testing
    "patch_size" : 16,
    "image_only" : False,
    # Image Transform Keys
    "train_transform_keys" : ["imagenet"],
    "val_transform_keys" : ["imagenet"],

    
    # Text Setting
    "text_encoder" : "google/electra-small-discriminator",
    # Train Text Encoder from Sratch if True
    "random_init_text_encoder" : False,
    # Manual Text Settings - Ignored unless random_init_text_encoder = True 
    "text_encoder_manual_configuration" : False,
    "text_encoder_hidden_size" : 192,
    "text_encoder_num_heads" : 4,
    "text_encoder_num_layers" : 12,
    "text_encoder_mlp_ratio" : 4,
    "text_encoder_drop_rate" : 0.1,
    "text_encoder_embedding_size" : 64,
    "max_text_len" : 40,
    "vocab_size" : 30522,

    
    # Cross Layer Settings
    "cross_layer_hidden_size" : 256,
    "num_cross_layers" : 6,
    "num_cross_layer_heads" : 4,
    "cross_layer_mlp_ratio" : 4,
    "cross_layer_drop_rate" : 0.1,
    
    # Freeze Module Parameter Settings
    "freeze_image_encoder" : False,
    "freeze_text_encoder" : False,
    "freeze_cross_modal_layers" : False,   
    
    # Pretraining Settings
    # Masked Language Mmodeling
    "whole_word_masking" : False, # note that whole_word_masking does not work for RoBERTa
    "mlm_prob" : 0.15,
    # Image-Text Matching
    "draw_false_image" : 1,
    "draw_false_text" : 0, 

    # Downstream Settings
    # Image-Text Recall
    "get_recall_metric" : False,
    # Visual Question Answering
    "vqav2_label_size" : 3129,

    # Optimizer Setting
    "optim_type" : "adamw",
    "learning_rate" : 1e-5,
    "weight_decay" : 0.01,
    "decay_power" : 1,
    "max_epoch" : 100,
    "max_steps" : 100000,
    "warmup_steps" : 10000,
    "end_lr" : 0,
    "lr_mult_head" : 5,  # multiply lr for downstream heads
    "lr_mult_cross_modal" : 5,  # multiply lr for the cross-modal module
    
    
    # PL Trainer Setting
    "fast_dev_run" : False,
    "val_check_interval" : 1.0,
    "test_only" : False,

    # below params varies with the environment
    "data_root" : 'data/arrow/', 
    "log_dir" : "result",
    "num_gpus" : 2,
    "num_nodes" : 1,
    "num_workers" : 12,
    "precision" : 32
}


In [5]:
model = RenaissanceTransformer(_config)
# dm = MTDataModule(_config, dist=False)
# dm.prepare_data()
# dm.setup('train')
model

Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-tiny-patch16-224 and are newly initialized: ['vit.pooler.dense.bias', 'vit.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RenaissanceTransformer(
  (encoder): TwoTowerEncoder(
    (image_encoder): ViTModel(
      (embeddings): ViTEmbeddings(
        (patch_embeddings): ViTPatchEmbeddings(
          (projection): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
        )
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (encoder): ViTEncoder(
        (layer): ModuleList(
          (0-11): 12 x ViTLayer(
            (attention): ViTAttention(
              (attention): ViTSelfAttention(
                (query): Linear(in_features=192, out_features=192, bias=True)
                (key): Linear(in_features=192, out_features=192, bias=True)
                (value): Linear(in_features=192, out_features=192, bias=True)
                (dropout): Dropout(p=0.0, inplace=False)
              )
              (output): ViTSelfOutput(
                (dense): Linear(in_features=192, out_features=192, bias=True)
                (dropout): Dropout(p=0.0, inplace=False)
              )
            )
   

In [6]:
for n, p in model.named_parameters():
    print(n)

encoder.image_encoder.embeddings.cls_token
encoder.image_encoder.embeddings.position_embeddings
encoder.image_encoder.embeddings.patch_embeddings.projection.weight
encoder.image_encoder.embeddings.patch_embeddings.projection.bias
encoder.image_encoder.encoder.layer.0.attention.attention.query.weight
encoder.image_encoder.encoder.layer.0.attention.attention.query.bias
encoder.image_encoder.encoder.layer.0.attention.attention.key.weight
encoder.image_encoder.encoder.layer.0.attention.attention.key.bias
encoder.image_encoder.encoder.layer.0.attention.attention.value.weight
encoder.image_encoder.encoder.layer.0.attention.attention.value.bias
encoder.image_encoder.encoder.layer.0.attention.output.dense.weight
encoder.image_encoder.encoder.layer.0.attention.output.dense.bias
encoder.image_encoder.encoder.layer.0.intermediate.dense.weight
encoder.image_encoder.encoder.layer.0.intermediate.dense.bias
encoder.image_encoder.encoder.layer.0.output.dense.weight
encoder.image_encoder.encoder.layer.

In [11]:
lr = model.hparams.config["learning_rate"]
wd = model.hparams.config["weight_decay"]

no_decay = [
    "bias",
    "LayerNorm.bias",
    "LayerNorm.weight",
    "norm.bias",
    "norm.weight",
    "norm1.bias",
    "norm1.weight",
    "norm2.bias",
    "norm2.weight",
]
head_names = ["vqa_classifier", "nlvr2_classifier", "mlm_score", "itm_score", "snli_classifier"]
cross_modal_names = ['cross_modal']
lr_mult_head = model.hparams.config["lr_mult_head"]
lr_mult_cross_modal = model.hparams.config["lr_mult_cross_modal"]
end_lr = model.hparams.config["end_lr"]
decay_power = model.hparams.config["decay_power"]
optim_type = model.hparams.config["optim_type"]

In [15]:
{
    "params": [
        n
        for n, p in model.named_parameters()
        if not any(nd in n for nd in no_decay)
        and not any(bb in n for bb in head_names)
        and not any(ht in n for ht in cross_modal_names)
    ],
    "weight_decay": wd,
    "lr": lr,
}

{'params': ['encoder.image_encoder.embeddings.cls_token',
  'encoder.image_encoder.embeddings.position_embeddings',
  'encoder.image_encoder.embeddings.patch_embeddings.projection.weight',
  'encoder.image_encoder.encoder.layer.0.attention.attention.query.weight',
  'encoder.image_encoder.encoder.layer.0.attention.attention.key.weight',
  'encoder.image_encoder.encoder.layer.0.attention.attention.value.weight',
  'encoder.image_encoder.encoder.layer.0.attention.output.dense.weight',
  'encoder.image_encoder.encoder.layer.0.intermediate.dense.weight',
  'encoder.image_encoder.encoder.layer.0.output.dense.weight',
  'encoder.image_encoder.encoder.layer.0.layernorm_before.weight',
  'encoder.image_encoder.encoder.layer.0.layernorm_after.weight',
  'encoder.image_encoder.encoder.layer.1.attention.attention.query.weight',
  'encoder.image_encoder.encoder.layer.1.attention.attention.key.weight',
  'encoder.image_encoder.encoder.layer.1.attention.attention.value.weight',
  'encoder.image_enco

In [17]:
{
    "params": [
        n
        for n, p in model.named_parameters()
        if any(nd in n for nd in no_decay)
        and not any(bb in n for bb in head_names)
        and not any(ht in n for ht in cross_modal_names)
    ],
    "weight_decay": 0.0,
    "lr": lr,
}

{'params': ['encoder.image_encoder.embeddings.patch_embeddings.projection.bias',
  'encoder.image_encoder.encoder.layer.0.attention.attention.query.bias',
  'encoder.image_encoder.encoder.layer.0.attention.attention.key.bias',
  'encoder.image_encoder.encoder.layer.0.attention.attention.value.bias',
  'encoder.image_encoder.encoder.layer.0.attention.output.dense.bias',
  'encoder.image_encoder.encoder.layer.0.intermediate.dense.bias',
  'encoder.image_encoder.encoder.layer.0.output.dense.bias',
  'encoder.image_encoder.encoder.layer.0.layernorm_before.bias',
  'encoder.image_encoder.encoder.layer.0.layernorm_after.bias',
  'encoder.image_encoder.encoder.layer.1.attention.attention.query.bias',
  'encoder.image_encoder.encoder.layer.1.attention.attention.key.bias',
  'encoder.image_encoder.encoder.layer.1.attention.attention.value.bias',
  'encoder.image_encoder.encoder.layer.1.attention.output.dense.bias',
  'encoder.image_encoder.encoder.layer.1.intermediate.dense.bias',
  'encoder.im

In [18]:
{
    "params": [
        n
        for n, p in model.named_parameters()
        if not any(nd in n for nd in no_decay)
        and any(bb in n for bb in head_names)
        and not any(ht in n for ht in cross_modal_names)
    ],
    "weight_decay": wd,
    "lr": lr * lr_mult_head,
}

{'params': ['mlm_score.transform.dense.weight',
  'mlm_score.decoder.weight',
  'itm_score.fc.weight'],
 'weight_decay': 0.01,
 'lr': 5e-05}

In [19]:
{
    "params": [
        n
        for n, p in model.named_parameters()
        if any(nd in n for nd in no_decay) and any(bb in n for bb in head_names)
        and not any(ht in n for ht in cross_modal_names)
    ],
    "weight_decay": 0.0,
    "lr": lr * lr_mult_head,
}

{'params': ['mlm_score.bias',
  'mlm_score.transform.dense.bias',
  'mlm_score.transform.LayerNorm.weight',
  'mlm_score.transform.LayerNorm.bias',
  'itm_score.fc.bias'],
 'weight_decay': 0.0,
 'lr': 5e-05}

In [20]:
{
    "params": [
        n
        for n, p in model.named_parameters()
        if not any(nd in n for nd in no_decay)
        and not any(bb in n for bb in head_names)
        and any(ht in n for ht in cross_modal_names)
    ],
    "weight_decay": wd,
    "lr": lr * lr_mult_cross_modal,
}

{'params': ['encoder.cross_modal_text_transform.weight',
  'encoder.cross_modal_image_transform.weight',
  'encoder.fusion_encoder.cross_modal_layers.0.visual_attention.att.query.weight',
  'encoder.fusion_encoder.cross_modal_layers.0.visual_attention.att.key.weight',
  'encoder.fusion_encoder.cross_modal_layers.0.visual_attention.att.value.weight',
  'encoder.fusion_encoder.cross_modal_layers.0.visual_attention.output.dense.weight',
  'encoder.fusion_encoder.cross_modal_layers.0.lang_self_att.self.query.weight',
  'encoder.fusion_encoder.cross_modal_layers.0.lang_self_att.self.key.weight',
  'encoder.fusion_encoder.cross_modal_layers.0.lang_self_att.self.value.weight',
  'encoder.fusion_encoder.cross_modal_layers.0.lang_self_att.output.dense.weight',
  'encoder.fusion_encoder.cross_modal_layers.0.visn_self_att.self.query.weight',
  'encoder.fusion_encoder.cross_modal_layers.0.visn_self_att.self.key.weight',
  'encoder.fusion_encoder.cross_modal_layers.0.visn_self_att.self.value.weight

In [21]:
{
    "params": [
        n
        for n, p in model.named_parameters()
        if any(nd in n for nd in no_decay)
        and not any(bb in n for bb in head_names)
        and any(ht in n for ht in cross_modal_names)
    ],
    "weight_decay": 0.0,
    "lr": lr * lr_mult_cross_modal,
}

{'params': ['encoder.cross_modal_text_transform.bias',
  'encoder.cross_modal_image_transform.bias',
  'encoder.fusion_encoder.cross_modal_layers.0.visual_attention.att.query.bias',
  'encoder.fusion_encoder.cross_modal_layers.0.visual_attention.att.key.bias',
  'encoder.fusion_encoder.cross_modal_layers.0.visual_attention.att.value.bias',
  'encoder.fusion_encoder.cross_modal_layers.0.visual_attention.output.dense.bias',
  'encoder.fusion_encoder.cross_modal_layers.0.visual_attention.output.LayerNorm.weight',
  'encoder.fusion_encoder.cross_modal_layers.0.visual_attention.output.LayerNorm.bias',
  'encoder.fusion_encoder.cross_modal_layers.0.lang_self_att.self.query.bias',
  'encoder.fusion_encoder.cross_modal_layers.0.lang_self_att.self.key.bias',
  'encoder.fusion_encoder.cross_modal_layers.0.lang_self_att.self.value.bias',
  'encoder.fusion_encoder.cross_modal_layers.0.lang_self_att.output.dense.bias',
  'encoder.fusion_encoder.cross_modal_layers.0.lang_self_att.output.LayerNorm.we

In [23]:
model.encoder.

TwoTowerEncoder(
  (image_encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=192, out_features=192, bias=True)
              (key): Linear(in_features=192, out_features=192, bias=True)
              (value): Linear(in_features=192, out_features=192, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=192, out_features=192, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=1